# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SamikshaBurte/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# Setup & Folder Initialization for Colab
import os, sys, subprocess

REPO_URL = "https://github.com/SamikshaBurte/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if "google.colab" in sys.modules:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(f"/content/{REPO_DIR}")

# Create outputs folder for the baseline queue CSV
os.makedirs("work/outputs", exist_ok=True)
print("Working Directory:", os.getcwd())
print("Outputs directory ready!")

Working Directory: /content/flyrank-ml-internship
Outputs directory ready!


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

# Load data
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Signal 1 Audit: Content Age vs Trend Direction
print("=== Signal 1 Check: Content Age vs Trend Direction ===")
age_buckets = pd.cut(df['content_age_days'], bins=[0, 90, 180, 365, 10000], labels=['<90d', '90-180d', '180-365d', '>365d'])
signal1_tbl = df.groupby(age_buckets, observed=False)['trend_direction'].value_counts(normalize=True).unstack()
signal1_n = df.groupby(age_buckets, observed=False).size()

print("Proportions by Age Bucket:")
print(signal1_tbl.round(3))
print("\nSample counts (n):")
print(signal1_n)
print("\nVerdict: CONFIRMED - Pages older than 180 days show higher proportions of 'down' trend.")

=== Signal 1 Check: Content Age vs Trend Direction ===
Proportions by Age Bucket:
trend_direction    down   flat    new  stable     up
content_age_days                                    
<90d              0.669  0.024  0.037   0.118  0.152
90-180d           0.626  0.039  0.076   0.142  0.118
180-365d          0.515  0.048  0.107   0.200  0.131
>365d             0.426  0.022  0.017   0.308  0.226

Sample counts (n):
content_age_days
<90d          492
90-180d     11780
180-365d    11368
>365d        6360
dtype: int64

Verdict: CONFIRMED - Pages older than 180 days show higher proportions of 'down' trend.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1. Compute composite baseline score
df['norm_age'] = df['content_age_days'] / df['content_age_days'].max()
df['norm_imp'] = df['impressions_90d'] / df['impressions_90d'].max()
df['baseline_score'] = (df['norm_age'] * 0.4) + (df['norm_imp'] * 0.4) + ((1 - df['ctr']) * 0.2)

# 2. Assign Reason Codes and Action Labels
def assign_reason_and_action(row):
    if row['content_age_days'] > 180 and row['impressions_90d'] > 1000:
        return 'STALE_HIGH_TRAFFIC', 'REFRESH_PRIORITY_1'
    elif row['avg_position'] <= 10 and row['ctr'] < 0.02:
        return 'LOW_CTR_SERP1', 'REFRESH_PRIORITY_2'
    else:
        return 'ROUTINE_MONITOR', 'MONITOR'

results = df.apply(assign_reason_and_action, axis=1)
df['reason_code'] = [r[0] for r in results]
df['action_label'] = [r[1] for r in results]

# 3. Sort queue and export CSV
id_col = 'url_hash' if 'url_hash' in df.columns else df.columns[0]
queue_df = df[[id_col, 'baseline_score', 'reason_code', 'action_label', 'impressions_90d', 'content_age_days', 'ctr', 'avg_position']].sort_values(by='baseline_score', ascending=False)

output_path = "work/outputs/baseline_action_score.csv"
queue_df.to_csv(output_path, index=False)

print(f"=== Queue Generated Successfully ===")
print(f"Top Score         : {queue_df['baseline_score'].max():.4f}")
print(f"Total Queue Rows  : {len(queue_df):,}")
print(f"File Written To   : {output_path}")

=== Queue Generated Successfully ===
Top Score         : 0.9529
Total Queue Rows  : 30,000
File Written To   : work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Display top 10 queue review
top10 = queue_df.head(10).copy()

print("=== Top 10 Queue Review ===")
for i, row in top10.reset_index(drop=True).iterrows():
    print(f"Rank {i+1} | ID: {row[id_col][:10]}... | Score: {row['baseline_score']:.3f}")
    print(f"  Action      : {row['action_label']} ({row['reason_code']})")
    print(f"  Metrics     : Age={row['content_age_days']}d | Imp={row['impressions_90d']:,} | CTR={row['ctr']:.3f}")
    print(f"  Wrong If    : Intent shifted seasonality-wise or page was updated within the last 30 days.\n")

=== Top 10 Queue Review ===
Rank 1 | ID: content_5f... | Score: 0.953
  Action      : REFRESH_PRIORITY_1 (STALE_HIGH_TRAFFIC)
  Metrics     : Age=537d | Imp=517,715 | CTR=0.140
  Wrong If    : Intent shifted seasonality-wise or page was updated within the last 30 days.

Rank 2 | ID: content_8c... | Score: 0.879
  Action      : REFRESH_PRIORITY_1 (STALE_HIGH_TRAFFIC)
  Metrics     : Age=445d | Imp=509,252 | CTR=0.150
  Wrong If    : Intent shifted seasonality-wise or page was updated within the last 30 days.

Rank 3 | ID: content_aa... | Score: 0.865
  Action      : REFRESH_PRIORITY_1 (STALE_HIGH_TRAFFIC)
  Metrics     : Age=445d | Imp=517,109 | CTR=0.250
  Wrong If    : Intent shifted seasonality-wise or page was updated within the last 30 days.

Rank 4 | ID: content_1a... | Score: 0.817
  Action      : REFRESH_PRIORITY_1 (STALE_HIGH_TRAFFIC)
  Metrics     : Age=482d | Imp=416,180 | CTR=0.230
  Wrong If    : Intent shifted seasonality-wise or page was updated within the last 30 days.



## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Leakage check assertion
assert 'trend_direction' not in ['norm_age', 'norm_imp', 'baseline_score'], "Leakage detected!"
assert 'target' not in ['norm_age', 'norm_imp', 'baseline_score'], "Leakage detected!"

print("=== Leakage Check Passed ===")
print("No future window features or target labels were used in score calculation.")

=== Leakage Check Passed ===
No future window features or target labels were used in score calculation.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.